# Study 03 — Hidden-State Classification and Sparse Autoencoders

**Research question.** Can classical classifiers exploit frozen language-model hidden states, and do PCA, ICA, Top-K sparse autoencoders (SAEs), or out-of-fold linear-probe features improve classification as labeled data increases?

Two distinct experiments are preserved from `Untitled8.ipynb`:

- **A · AG News:** 4 classes, BERT and E5, document-level SAE, scikit-learn classifiers, 20,000 training-pool documents, 2,000 test documents.
- **B · 20 Newsgroups:** 20 classes, BERT and E5, token-level Top-K SAE with mean/max latent pooling, GPU kNN/logistic regression/XGBoost, and a held-out test subset.

The experiments use **independent original model pipelines** (different text lengths, SAE designs, classification implementations, and label budgets). Compare feature methods *within* each experiment; do not interpret A-versus-B accuracy differences as a controlled benchmark.

**Important:** This is an exploratory experiment. Unlabeled representations are fitted on the training pool, while final classifiers use only labeled subsets. The held-out test set is used for final evaluation, not for fitting transforms. Results should not be selected based on test performance without a separate validation set. The original numerical methods and hyperparameters are retained, except explicitly marked robustness or GPU changes.


## 0 · Colab setup

Choose **Runtime → Change runtime type → T4 GPU** (or another GPU). Run cells in order. Models/datasets download automatically; CPU fallbacks are available but substantially slower. Some preprocessing (PCA, ICA, and decision trees) intentionally runs on CPU.

In [ ]:
%pip -q install "transformers>=4.43,<5" "datasets>=2.20,<5" "huggingface-hub>=0.30,<1" "scikit-learn>=1.3,<2" "xgboost>=2,<4" matplotlib pandas

In [ ]:
import os, random, warnings, time, gc
import numpy as np, torch
SEED_GLOBAL = 0
random.seed(SEED_GLOBAL); np.random.seed(SEED_GLOBAL); torch.manual_seed(SEED_GLOBAL)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_GLOBAL)
    print("GPU:", torch.cuda.get_device_name(0), "| total memory (GB):", round(torch.cuda.get_device_properties(0).total_memory/2**30, 1))
else:
    print("WARNING: No CUDA GPU detected. Go to Runtime → Change runtime type → GPU.")
OUTPUT_DIR = "/content/study03_results" if os.path.isdir('/content') else "study03_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## A · AG News — document-level SAE

**Hypothesis:** sparse reconstruction features might provide a useful alternative to raw standardized vectors, PCA/ICA or linear-probe class probabilities for classical classifiers.

**Preserved settings:** 20k unlabeled training-pool sentences; 2k held-out test; label budgets 100/500/2000/6000; BERT-base and E5-small-v2; 2,048 SAE latents with top-32 activations; PCA/ICA each 64 components. SAE and frozen encoders use GPU where available; scikit-learn classifiers run on CPU.

**Potential caveat:** every feature representation is fitted on the complete *unlabeled training pool* (transductive for the labeled-budget comparison, not for the held-out test). The out-of-fold probe helps avoid training-set self-prediction. No model-selection validation split was provided in the original experiment.

In [ ]:
# Standalone Colab notebook script. Run in a fresh notebook:
#   !pip install -q transformers datasets scikit-learn
# Question: can classic ML (kNN, decision trees, ...) use hidden states, and do
# PCA / ICA / sparse autoencoder / probe features help trees?

import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, FastICA
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

QUICK = False          # True: fewer sizes, skips gradient boosting (much faster)
N_POOL = 20000         # unlabeled sentences used to fit PCA / ICA / SAE
N_TEST = 2000
LABEL_SIZES = [100, 500, 2000, 6000] if not QUICK else [100, 500, 2000]
SAE_LATENTS = 2048
SAE_K = 32             # active latents per sentence
SEED = 0
device = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = {
    # name: (hub id, prefix, hidden-state index, number of layers tells us what 'last' is)
    "bert": ("bert-base-uncased", "", 11),
    "e5": ("intfloat/e5-small-v2", "query: ", -1),
}


def load_data():
    from datasets import load_dataset
    train = load_dataset("fancyzhx/ag_news", split="train").shuffle(seed=SEED).select(range(N_POOL))
    test = load_dataset("fancyzhx/ag_news", split="test").shuffle(seed=SEED).select(range(N_TEST))
    return (list(train["text"]), np.array(train["label"]),
            list(test["text"]), np.array(test["label"]))


@torch.no_grad()
def extract(model_id, prefix, layer, sentences, batch_size=64):
    from transformers import AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id).to(device).eval()
    chunks = []
    for i in range(0, len(sentences), batch_size):
        enc = tok([prefix + s for s in sentences[i:i + batch_size]], padding=True,
                  truncation=True, max_length=64, return_tensors="pt").to(device)
        hidden = model(**enc, output_hidden_states=True).hidden_states[layer]
        mask = enc["attention_mask"].unsqueeze(-1).float()
        chunks.append(((hidden * mask).sum(1) / mask.sum(1)).float().cpu())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return torch.cat(chunks).numpy()


class TopKSAE(nn.Module):
    """Sparse autoencoder: only the k largest latents stay active per example."""

    def __init__(self, d_in, d_latent, k):
        super().__init__()
        self.k = k
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.enc = nn.Linear(d_in, d_latent)
        self.dec = nn.Linear(d_latent, d_in, bias=False)

    def encode(self, x):
        z = torch.relu(self.enc(x - self.b_dec))
        top = torch.topk(z, self.k, dim=1)
        return torch.zeros_like(z).scatter_(1, top.indices, top.values)

    def forward(self, x):
        z = self.encode(x)
        return self.dec(z) + self.b_dec, z


def train_sae(X, epochs=40, batch_size=256, lr=1e-3):
    torch.manual_seed(SEED)
    sae = TopKSAE(X.shape[1], SAE_LATENTS, SAE_K).to(device)
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    data = torch.tensor(X, dtype=torch.float32, device=device)
    for epoch in range(epochs):
        perm = torch.randperm(len(data), device=device)
        total = 0.0
        for i in range(0, len(data), batch_size):
            batch = data[perm[i:i + batch_size]]
            recon, _ = sae(batch)
            loss = ((recon - batch) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(batch)
        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"  SAE epoch {epoch:2d}  reconstruction MSE {total / len(data):.4f}")
    sae.eval()
    return sae


def sae_transform(sae, X):
    with torch.no_grad():
        return sae.encode(torch.tensor(X, dtype=torch.float32, device=device)).cpu().numpy()


def classifiers():
    c = {
        "logreg": LogisticRegression(max_iter=500, C=0.1),
        "kNN": KNeighborsClassifier(n_neighbors=15, metric="cosine", algorithm="brute"),
        "tree": DecisionTreeClassifier(max_depth=10, min_samples_leaf=2, random_state=SEED),
        "forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED),
    }
    if not QUICK:
        c["boosting"] = HistGradientBoostingClassifier(max_iter=100, random_state=SEED)
    return c


def run_model(name, X_pool, y_pool, X_test, y_test):
    scaler = StandardScaler().fit(X_pool)
    P, T = scaler.transform(X_pool), scaler.transform(X_test)

    t0 = time.time()
    pca = PCA(n_components=64, whiten=True, random_state=SEED).fit(P)
    print(f"[{name}] PCA fitted ({time.time() - t0:.0f}s), variance kept: {pca.explained_variance_ratio_.sum():.2f}")
    t0 = time.time()
    ica = FastICA(n_components=64, whiten="unit-variance", max_iter=500, random_state=SEED).fit(P[:10000])
    print(f"[{name}] ICA fitted ({time.time() - t0:.0f}s)")
    print(f"[{name}] training sparse autoencoder on {len(P)} unlabeled sentences")
    sae = train_sae(P)

    spaces = {
        "raw": (P, T),
        "pca64": (pca.transform(P), pca.transform(T)),
        "ica64": (ica.transform(P), ica.transform(T)),
        "sae": (sae_transform(sae, P), sae_transform(sae, T)),
    }
    sparsity = (spaces["sae"][0] > 0).mean()
    dead = ((spaces["sae"][0] > 0).sum(0) == 0).mean()
    print(f"[{name}] SAE: {sparsity:.3%} of latents active, {dead:.1%} never active")

    rows = []
    for n in LABEL_SIZES:
        idx = np.arange(n)  # pool is already shuffled
        y = y_pool[idx]
        for space, (Ptr, Tst) in spaces.items():
            for cname, clf in classifiers().items():
                clf.fit(Ptr[idx], y)
                rows.append((name, n, space, cname, (clf.predict(Tst) == y_test).mean()))
        # Probe features: class scores from a linear probe. Out-of-fold scores for the
        # training set so the tree does not see scores fitted on its own labels.
        probe = LogisticRegression(max_iter=500, C=0.1)
        oof = cross_val_predict(probe, P[idx], y, cv=5, method="predict_proba")
        probe.fit(P[idx], y)
        for cname, clf in classifiers().items():
            clf.fit(oof, y)
            rows.append((name, n, "probe", cname, (clf.predict(probe.predict_proba(T)) == y_test).mean()))
        print(f"[{name}] finished {n} labeled examples")
    return pd.DataFrame(rows, columns=["model", "n_labeled", "features", "classifier", "accuracy"])


def report(results):
    for name, df in results.groupby("model"):
        table = df.pivot_table(index=["features", "classifier"], columns="n_labeled", values="accuracy")
        print(f"\n=== {name}: test accuracy by number of labeled examples ===")
        print(table.round(3).to_string())

    models = results["model"].unique()
    clfs = results["classifier"].unique()
    fig, axes = plt.subplots(len(models), len(clfs), figsize=(4 * len(clfs), 3.6 * len(models)), squeeze=False)
    for r, m in enumerate(models):
        for c, cname in enumerate(clfs):
            ax = axes[r][c]
            sub = results[(results["model"] == m) & (results["classifier"] == cname)]
            for feat, g in sub.groupby("features"):
                ax.plot(g["n_labeled"], g["accuracy"], marker="o", label=feat)
            ax.set_xscale("log")
            ax.set_title(f"{m} / {cname}")
            ax.set_xlabel("Labeled examples")
            ax.set_ylabel("Test accuracy")
            ax.set_ylim(0.2, 1.0)
            if r == 0 and c == 0:
                ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "ag_news_comparison.png"), dpi=180, bbox_inches="tight")
    plt.show()


def main():
    texts, labels, test_texts, test_labels = load_data()
    all_results = []
    for name, (model_id, prefix, layer) in MODELS.items():
        print(f"\nExtracting features with {name} ...")
        X_pool = extract(model_id, prefix, layer, texts)
        X_test = extract(model_id, prefix, layer, test_texts)
        all_results.append(run_model(name, X_pool, labels, X_test, test_labels))
    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv(os.path.join(OUTPUT_DIR, "ag_news_results.csv"), index=False)
    report(combined)
    return combined





### A · Execute and export results

This runs the **full original** AG News experiment (not the quick mode). It may take considerable time because it trains many CPU-based classical classifiers for multiple label budgets.

In [ ]:
ag_news_results = main()
print("AG News rows:", len(ag_news_results))
display(ag_news_results.groupby(["model","features","classifier"],as_index=False)["accuracy"].max().sort_values("accuracy",ascending=False).head(15))

## B · 20 Newsgroups — token-level Top-K SAE

**Research question:** can the same frozen language-model states yield more informative features when the SAE is trained on *individual token activations*, then pooled back to the document with `mean` or `max`?

**Preserved settings:** original train/test split; up to 4,000 sampled test documents, label budgets 200/1,000/4,000; max 256 input tokens; 64 tokens per train document for SAE fitting; 6,144 SAE latents, top-32, eight epochs; BERT and E5. The original FP16 autocast for extraction is retained.

**GPU use:** frozen transformer encoding, SAE, logistic regression, kNN, and GPU XGBoost. PCA, ICA, and sklearn decision tree run on CPU. Beware that the 6,144-dimensional SAE features and their repeated classifiers can be memory intensive.

**Methodological note:** The two-stage token sampling/SAE fitting uses *training-pool documents only*. The test documents are processed using the frozen encoder and the pretrained SAE. A strong result must still be replicated with multiple seeds or independent validation data.

In [ ]:
# Standalone Colab script (GPU runtime recommended). Fresh notebook:
#   !pip install -q transformers scikit-learn xgboost
#
# 20 topics (20 Newsgroups) instead of 4. Everything heavy runs on the GPU:
# feature extraction (fp16, length-sorted batches), a token-level sparse
# autoencoder, logistic regression, kNN, and XGBoost (boosting / random forest).
# Only the single decision tree, PCA and ICA run on the CPU.

import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, FastICA
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier, XGBRFClassifier

warnings.filterwarnings("ignore", category=UserWarning)

QUICK = False
SEED = 0
MAX_TOKENS = 256            # per document
N_TEST = 4000
LABEL_SIZES = [200, 1000, 4000] if not QUICK else [200, 1000]
TOKENS_PER_DOC = 64         # token vectors sampled per document to train the SAE
SAE_LATENTS = 6144
SAE_K = 32
SAE_EPOCHS = 8
device = "cuda" if torch.cuda.is_available() else "cpu"
use_cuda = device == "cuda"

MODELS = {
    "bert": ("bert-base-uncased", "", 11),
    "e5": ("intfloat/e5-small-v2", "query: ", -1),
}


def log(msg, t0=[time.time()]):
    print(f"[{time.time() - t0[0]:6.0f}s] {msg}", flush=True)


# ----------------------------------------------------------------- data
def load_data():
    from sklearn.datasets import fetch_20newsgroups
    out = []
    for subset in ("train", "test"):
        d = fetch_20newsgroups(subset=subset, remove=("headers", "footers", "quotes"))
        keep = [i for i, t in enumerate(d.data) if len(t.split()) >= 5]
        out.append(([d.data[i] for i in keep], np.array(d.target)[keep], list(d.target_names)))
    rng = np.random.default_rng(SEED)
    (tr_x, tr_y, names), (te_x, te_y, _) = out
    p = rng.permutation(len(tr_x))
    q = rng.permutation(len(te_x))[:N_TEST]
    return ([tr_x[i] for i in p], tr_y[p], [te_x[i] for i in q], te_y[q], names)


# -------------------------------------------------------- sparse autoencoder
class TopKSAE(nn.Module):
    def __init__(self, d_in, d_latent, k):
        super().__init__()
        self.k = k
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.enc = nn.Linear(d_in, d_latent)
        self.dec = nn.Linear(d_latent, d_in, bias=False)
        with torch.no_grad():
            self.dec.weight.copy_(F.normalize(torch.randn_like(self.dec.weight), dim=0))
            self.enc.weight.copy_(self.dec.weight.T)
            self.enc.bias.zero_()

    def encode(self, x):
        z = torch.relu(self.enc(x - self.b_dec))
        top = torch.topk(z, self.k, dim=1)
        return torch.zeros_like(z).scatter_(1, top.indices, top.values)

    def forward(self, x):
        z = self.encode(x)
        return self.dec(z) + self.b_dec, z

    @torch.no_grad()
    def normalize_decoder(self):
        self.dec.weight.copy_(F.normalize(self.dec.weight, dim=0))


def train_sae(tokens, epochs=SAE_EPOCHS, batch_size=2048, lr=5e-4, latents=SAE_LATENTS, k=SAE_K):
    """tokens: standardized float tensor (N, d) on the device."""
    torch.manual_seed(SEED)
    sae = TopKSAE(tokens.shape[1], latents, k).to(device)
    sae.b_dec.data = tokens.mean(0)
    opt = torch.optim.Adam(sae.parameters(), lr=lr)
    n = len(tokens)
    for epoch in range(epochs):
        fired = torch.zeros(latents, dtype=torch.bool, device=device)
        perm = torch.randperm(n, device=device)
        total = 0.0
        for i in range(0, n, batch_size):
            x = tokens[perm[i:i + batch_size]]
            recon, z = sae(x)
            loss = ((recon - x) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            sae.normalize_decoder()
            fired |= (z > 0).any(0)
            total += loss.item() * len(x)
        dead = (~fired).sum().item()
        log(f"  SAE epoch {epoch}: MSE {total / n:.4f}, dead latents {dead}/{latents}")
        if dead and epoch < epochs - 1:
            with torch.no_grad():  # revive dead latents on the worst-reconstructed tokens
                sample = tokens[torch.randperm(n, device=device)[:20000]]
                recon, _ = sae(sample)
                err = ((recon - sample) ** 2).sum(1)
                worst = err.topk(min(dead, len(sample))).indices
                direction = F.normalize(sample[worst] - recon[worst], dim=1)
                idx = (~fired).nonzero().squeeze(1)[:len(worst)]
                sae.dec.weight[:, idx] = direction.T
                sae.enc.weight[idx] = direction
                sae.enc.bias[idx] = 0
    sae.eval()
    return sae


def pool_latents(z, doc_idx, n_docs):
    """Mean- and max-pool token latents (T, m) into per-document vectors."""
    m = z.shape[1]
    total = torch.zeros(n_docs, m, device=z.device).index_add_(0, doc_idx, z)
    counts = torch.bincount(doc_idx, minlength=n_docs).clamp(min=1).unsqueeze(1).float()
    peak = torch.zeros(n_docs, m, device=z.device).scatter_reduce_(
        0, doc_idx.unsqueeze(1).expand(-1, m), z, reduce="amax", include_self=True)
    return total / counts, peak


# ------------------------------------------------------------ extraction
@torch.no_grad()
def encode_docs(model, tok, docs, prefix, layer, sae=None, mu=None, sd=None,
                tokens_per_doc=0, rng=None):
    order = np.argsort([len(d) for d in docs])
    batch = 32 if sae is not None else 64
    pooled, sae_mean, sae_max = [None] * len(docs), [None] * len(docs), [None] * len(docs)
    tok_vecs, tok_ids = [], []
    special = torch.tensor(sorted(set(tok.all_special_ids)), device=device)
    for start in range(0, len(docs), batch):
        ids = order[start:start + batch]
        enc = tok([prefix + docs[i] for i in ids], padding=True, truncation=True,
                  max_length=MAX_TOKENS, return_tensors="pt").to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_cuda):
            h = model(**enc, output_hidden_states=True).hidden_states[layer]
        h = h.float()
        mask = enc["attention_mask"].bool()
        mean = (h * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
        for j, i in enumerate(ids):
            pooled[i] = mean[j].cpu()
        if tokens_per_doc:
            real = mask & ~torch.isin(enc["input_ids"], special)
            for j in range(len(ids)):
                pos = real[j].nonzero().squeeze(1)
                if len(pos) > tokens_per_doc:
                    pos = pos[torch.from_numpy(rng.choice(len(pos), tokens_per_doc, replace=False)).to(device)]
                tok_vecs.append(h[j, pos].half().cpu())
                tok_ids.append(enc["input_ids"][j, pos].cpu())
        if sae is not None:
            doc_idx = torch.arange(len(ids), device=device).unsqueeze(1).expand_as(mask)[mask]
            z = sae.encode((h[mask] - mu) / sd)
            mean_z, max_z = pool_latents(z, doc_idx, len(ids))
            for j, i in enumerate(ids):
                sae_mean[i], sae_max[i] = mean_z[j].cpu(), max_z[j].cpu()
    out = {"mean": torch.stack(pooled).numpy()}
    if tokens_per_doc:
        out["tokens"], out["token_ids"] = torch.cat(tok_vecs), torch.cat(tok_ids)
    if sae is not None:
        out["sae_mean"], out["sae_max"] = torch.stack(sae_mean).numpy(), torch.stack(sae_max).numpy()
    return out


def show_latents(sae, tokens, token_ids, tok, mu, sd, n_show=8):
    with torch.no_grad():
        sample = tokens[:100000].float().to(device)
        z = sae.encode((sample - mu) / sd)
    alive = (z > 0).sum(0).nonzero().squeeze(1).cpu().numpy()
    print(f"  {len(alive)}/{z.shape[1]} latents fire on a 100k-token sample")
    rng = np.random.default_rng(SEED)
    for lat in rng.choice(alive, size=min(n_show, len(alive)), replace=False):
        top = z[:, lat].topk(min(30, len(z))).indices.cpu()
        words = tok.convert_ids_to_tokens(token_ids[:100000][top])
        counts = pd.Series(words).value_counts().head(6)
        print(f"  latent {lat:5d}: " + ", ".join(f"{w} x{c}" for w, c in counts.items()))


# ------------------------------------------------------------ classifiers
def stratified_order(y, rng):
    per_class = [rng.permutation(np.where(y == c)[0]) for c in np.unique(y)]
    longest = max(len(p) for p in per_class)
    order = []
    for r in range(longest):
        for p in per_class:
            if r < len(p):
                order.append(p[r])
    return np.array(order)


def gpu_logreg_probs(Xtr, ytr, Xte, n_classes, C=0.1, iters=100):
    Xtr, Xte = (torch.as_tensor(a, dtype=torch.float32, device=device) for a in (Xtr, Xte))
    ytr = torch.as_tensor(ytr, device=device)
    W = torch.zeros(Xtr.shape[1], n_classes, device=device, requires_grad=True)
    b = torch.zeros(n_classes, device=device, requires_grad=True)
    lam = 1.0 / (C * len(Xtr))
    opt = torch.optim.LBFGS([W, b], lr=1, max_iter=iters, history_size=20, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(Xtr @ W + b, ytr) + 0.5 * lam * (W ** 2).sum()
        loss.backward()
        return loss

    opt.step(closure)
    with torch.no_grad():
        return torch.softmax(Xte @ W + b, dim=1).cpu().numpy()


def gpu_knn_predict(Xtr, ytr, Xte, n_classes, k=15):
    Xtr = F.normalize(torch.as_tensor(Xtr, dtype=torch.float32, device=device), dim=1)
    Xte = F.normalize(torch.as_tensor(Xte, dtype=torch.float32, device=device), dim=1)
    ytr = torch.as_tensor(ytr, device=device)
    nn_idx = (Xte @ Xtr.T).topk(k, dim=1).indices
    votes = torch.zeros(len(Xte), n_classes, device=device)
    votes.scatter_add_(1, ytr[nn_idx], torch.ones_like(nn_idx, dtype=torch.float32))
    return votes.argmax(1).cpu().numpy()


def xgb_kwargs():
    return dict(tree_method="hist", device="cuda" if use_cuda else "cpu", verbosity=0)


def evaluate_space(Xtr, ytr, Xte, yte, n_classes):
    """Accuracy of every classifier on one feature space."""
    acc = {}
    acc["logreg"] = (gpu_logreg_probs(Xtr, ytr, Xte, n_classes).argmax(1) == yte).mean()
    acc["kNN"] = (gpu_knn_predict(Xtr, ytr, Xte, n_classes) == yte).mean()
    tree = DecisionTreeClassifier(max_depth=10, min_samples_leaf=2, random_state=SEED).fit(Xtr, ytr)
    acc["tree"] = (tree.predict(Xte) == yte).mean()
    forest = XGBRFClassifier(n_estimators=100, max_depth=8, subsample=0.63,
                             colsample_bynode=0.3, learning_rate=1.0, **xgb_kwargs()).fit(Xtr, ytr)
    acc["forest"] = (forest.predict(Xte) == yte).mean()
    if not QUICK:
        boost = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.2,
                              colsample_bytree=0.5, **xgb_kwargs()).fit(Xtr, ytr)
        acc["boosting"] = (boost.predict(Xte) == yte).mean()
    return acc


def probe_features(Ptr, ytr, Pte, n_classes, rng, folds=5):
    """Class probabilities from a linear probe; out-of-fold on the training set."""
    perm = rng.permutation(len(Ptr))
    oof = np.zeros((len(Ptr), n_classes), dtype=np.float32)
    for f in range(folds):
        held = perm[f::folds]
        rest = np.setdiff1d(perm, held)
        oof[held] = gpu_logreg_probs(Ptr[rest], ytr[rest], Ptr[held], n_classes)
    return oof, gpu_logreg_probs(Ptr, ytr, Pte, n_classes)


def zscore(train, test):
    mu, sd = train.mean(0), train.std(0) + 1e-6
    return (train - mu) / sd, (test - mu) / sd


def run_model(name, feats_pool, feats_test, y_pool, y_test, n_classes):
    rng = np.random.default_rng(SEED)
    P, T = zscore(feats_pool["mean"], feats_test["mean"])
    log(f"[{name}] PCA / ICA")
    pca = PCA(n_components=64, whiten=True, random_state=SEED).fit(P)
    ica = FastICA(n_components=64, whiten="unit-variance", max_iter=500, random_state=SEED).fit(P[:8000])
    spaces = {
        "raw": (P, T),
        "pca64": (pca.transform(P), pca.transform(T)),
        "ica64": (ica.transform(P), ica.transform(T)),
        "sae_mean": zscore(feats_pool["sae_mean"], feats_test["sae_mean"]),
        "sae_max": zscore(feats_pool["sae_max"], feats_test["sae_max"]),
    }
    order = stratified_order(y_pool, rng)
    rows = []
    for n in LABEL_SIZES:
        idx = order[:n]
        y = y_pool[idx]
        for space, (Ptr, Tst) in spaces.items():
            t0 = time.time()
            for clf, a in evaluate_space(Ptr[idx], y, Tst, y_test, n_classes).items():
                rows.append((name, n, space, clf, a))
            log(f"[{name}] n={n} {space}: {time.time() - t0:.0f}s")
        Otr, Ote = probe_features(P[idx], y, T, n_classes, rng)
        for clf, a in evaluate_space(Otr, y, Ote, y_test, n_classes).items():
            rows.append((name, n, "probe", clf, a))
    return pd.DataFrame(rows, columns=["model", "n_labeled", "features", "classifier", "accuracy"])


def report(results, n_classes):
    for name, df in results.groupby("model"):
        table = df.pivot_table(index=["features", "classifier"], columns="n_labeled", values="accuracy")
        print(f"\n=== {name}: test accuracy ({n_classes} classes, chance {1 / n_classes:.2f}) ===")
        print(table.round(3).to_string())
    models, clfs = results["model"].unique(), results["classifier"].unique()
    fig, axes = plt.subplots(len(models), len(clfs), figsize=(4 * len(clfs), 3.6 * len(models)), squeeze=False)
    for r, m in enumerate(models):
        for c, clf in enumerate(clfs):
            ax = axes[r][c]
            sub = results[(results["model"] == m) & (results["classifier"] == clf)]
            for feat, g in sub.groupby("features"):
                ax.plot(g["n_labeled"], g["accuracy"], marker="o", label=feat)
            ax.set_xscale("log")
            ax.set_title(f"{m} / {clf}")
            ax.set_xlabel("Labeled examples")
            ax.set_ylabel("Test accuracy")
            ax.set_ylim(0, 1)
            if r == 0 and c == 0:
                ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "newsgroups_comparison.png"), dpi=180, bbox_inches="tight")
    plt.show()


def main():
    from transformers import AutoModel, AutoTokenizer
    log(f"device: {device}")
    pool_x, pool_y, test_x, test_y, class_names = load_data()
    n_classes = len(class_names)
    log(f"{len(pool_x)} pool docs, {len(test_x)} test docs, {n_classes} classes")
    results = []
    for name, (model_id, prefix, layer) in MODELS.items():
        log(f"=== {name}: loading model")
        tok = AutoTokenizer.from_pretrained(model_id)
        model = AutoModel.from_pretrained(model_id).to(device).eval()
        rng = np.random.default_rng(SEED)

        log(f"[{name}] pass 1: hidden states + token sample for the SAE")
        first = encode_docs(model, tok, pool_x, prefix, layer, tokens_per_doc=TOKENS_PER_DOC, rng=rng)
        tokens = first["tokens"].float().to(device)
        mu, sd = tokens.mean(0), tokens.std(0) + 1e-6
        log(f"[{name}] training SAE on {len(tokens)} token vectors")
        sae = train_sae((tokens - mu) / sd)
        show_latents(sae, first["tokens"], first["token_ids"], tok, mu, sd)
        del tokens, first
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        log(f"[{name}] pass 2: documents with SAE pooling")
        feats_pool = encode_docs(model, tok, pool_x, prefix, layer, sae=sae, mu=mu, sd=sd)
        feats_test = encode_docs(model, tok, test_x, prefix, layer, sae=sae, mu=mu, sd=sd)
        del model, sae
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        results.append(run_model(name, feats_pool, feats_test, pool_y, test_y, n_classes))
    combined = pd.concat(results, ignore_index=True)
    combined.to_csv(os.path.join(OUTPUT_DIR, "newsgroups_results.csv"), index=False)
    report(combined, n_classes)
    return combined





### B · Execute and export results

This runs the **full original** Newsgroups experiment. GPU memory demand can be high; use a fresh runtime and close other notebooks if an out-of-memory error occurs.

In [ ]:
newsgroups_results = main()
print("20 Newsgroups rows:", len(newsgroups_results))
display(newsgroups_results.groupby(["model","features","classifier"],as_index=False)["accuracy"].max().sort_values("accuracy",ascending=False).head(15))

## Results and interpretation

Plots and result CSVs above are computed in this session; **the notebook intentionally does not insert historical results as if they had been rerun**. After running:

1. Compare feature methods at each **fixed labeled budget, model and classifier** (rather than mixing maximum accuracies from different settings).
2. Distinguish *representation improvement* from *better supervised classifier capacity*, especially for out-of-fold probe features.
3. Report any failed PCA/ICA convergence or class-count errors rather than silently dropping configurations.
4. The held-out test set must not be used to tune SAE sparsity, layer choices, hyperparameters, or model selection. Add a validation split before repeated experimentation.
5. A strong comparison should include repeated seeds and uncertainty intervals; this reproduction preserves the original single-seed study.


In [ ]:
from pathlib import Path
from IPython.display import display
for p in sorted(Path(OUTPUT_DIR).glob("*")):
    print(p.name, f"({p.stat().st_size/1024:.1f} KiB)")
import zipfile
zip_path = "/content/study03_results.zip" if os.path.isdir('/content') else "study03_results.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in Path(OUTPUT_DIR).glob("*"):
        z.write(p,p.name)
print("Saved:",zip_path)
if os.path.isdir('/content'):
    from google.colab import files
    files.download(zip_path)
